# Demo: Create Agents to Research and Write an Article

This notebook demonstrates how to create a simple multi-agent multi-agent system using the crewAI framework.

## Preparing environment

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew, LLM

In [3]:
from IPython.display import Markdown

In [4]:
AGENT_ALLOW_DELEGATION = False
AGENT_VERBOSE = True
AGENT_TRACING = True

## Prepare locally hosted LLM

In [5]:
ollama_llm = LLM(
    model="ollama/mistral:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_:

```
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:

```
varname = """line 1 of text
             line 2 of text
          """
```

is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [6]:
planner = Agent(
    role="Connect Planner",
    goal="Plan engaging and facturally accurate content on {topic}",
    backstory="You're working on planning a blog article"
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Writer

In [7]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provided by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

### Agent: Editor

In [8]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=ollama_llm,
    allow_delegation=AGENT_ALLOW_DELEGATION,
    verbose=AGENT_VERBOSE
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [9]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
                    "with an outline, audience analysis, "
                    "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [10]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
                    "in markdown format, ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [11]:
edit = Task(
    description=(
        "Proofread the given blog post for "
        "grammatical errors and alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: _For this simple example_, the tasks will be performed sequentially (i.e. they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=True` allows you to see execution logs.



In [12]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=AGENT_VERBOSE,
    tracing=AGENT_TRACING
)

## Running the Crew

In [13]:
from time import perf_counter

In [14]:
start = perf_counter()
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})
end = perf_counter()
print(f"\n\n{'='*100}\nTasks are finished in {end - start} seconds.\n{'='*100}\n\n")

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d9a45f84-7a8a-4927-855e-7a8405b9d152                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Connect Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Content Plan Document for "Artificial Intelligence: Latest Trends, Key Players, and Noteworthy News"           │
│                                                                                                                 │
│  I. Introduction                                                                                                │
│     A. Brief explanation of Artificial Intelligence (AI) and its relevance today                                │
│     B. Hook the audience with a recent, intriguing AI-related news story or trend                               │
│     C. State the purpose of the article: To provide readers with an understanding of the latest trends, key     │
│  players, and noteworthy news in the field of Artificial Intelligence                                           │
│                                                                                                                 │
│  II. Latest Trends in AI                                                                                        │
│     A. Explanation of the current state-of-the-art in AI research                                               │
│        1. Machine Learning (ML)                                                                                 │
│           a. Deep learning                                                                                      │
│           b. Reinforcement learning                                                                             │
│        2. Natural Language Processing (NLP)                                                                     │
│        3. Computer Vision                                                                                       │
│     B. Emerging trends and applications in various industries: Healthcare, Finance, Transportation, Retail,     │
│  etc.                                                                                                           │
│     C. Discussion of AI ethics and regulations, such as data privacy and bias mitigation                        │
│                                                                                                                 │
│  III. Key Players in the AI Industry                                                                            │
│     A. Overview of leading companies developing AI technologies                                                 │
│        1. Google DeepMind                                                                                       │
│        2. IBM Watson                                                                                            │
│        3. Microsoft Azure                                                                                       │
│        4. Amazon Web Services (AWS)                                                                             │
│        5. NVIDIA                                                                                                │
│     B. Analysis of notable startups and research institutions contributing to the field                         │
│     C. Discussion on collaborations, partnerships, and acquisitions among these players                         │
│                                                                                                                 │
│  IV. Noteworthy News in AI                             

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1202ecb9-ad05-4b83-a299-5560e21d44ec                                                                     │
│  Agent: Connect Planner                                                                                         │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Writer                                                                                          │
│                                                                                                                 │
│  Task: 1. Use the content plan to craft a compelling blog post on Artificial Intelligence.                      │
│  2. Incorporate SEO keywords naturally.                                                                         │
│  3. Sections/Subtitles are properly named in an engaging manner.                                                │
│  4. Ensure the post is structured with an engaging introduction, insightful body, and a summarizing             │
│  conclusion.                                                                                                    │
│  5. Proofread for grammatical errors and alignment with the brand's voice.                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here's an edited version of the blog post that aligns with the brand's voice and follows journalistic best     │
│  practices:                                                                                                     │
│                                                                                                                 │
│  ```markdown                                                                                                    │
│  # Unveiling the Latest in Artificial Intelligence: Trends, Pioneers, and News                                  │
│                                                                                                                 │
│  Welcome to our latest exploration into the dynamic world of Artificial Intelligence (AI)! As AI continues to   │
│  transform industries and reshape our daily lives, we delve deeper into the exciting advancements, key          │
│  players, and noteworthy news shaping this rapidly evolving field.                                              │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  Artificial Intelligence, a marvel of human innovation, is becoming increasingly integral to our society. With  │
│  breakthroughs in quantum computing, AI's potential has never been greater. Join us as we unravel the latest    │
│  trends, pioneers, and news in the realm of Artificial Intelligence.                                            │
│                                                                                                                 │
│  ## Latest Trends in AI                                                                                         │
│                                                                                                                 │
│  The cutting-edge research in AI is nothing short of breathtaking! From Machine Learning (ML) to Natural        │
│  Language Processing (NLP), Computer Vision, and beyond, here's a glimpse:                                      │
│                                                                                                                 │
│  1. **Machine Learning**: The subfields of Deep learning and Reinforcement learning are revolutionizing the     │
│  way machines learn from data.                                                                                  │
│  2. **Natural Language Processing (NLP)**: Recent advancements in NLP have made AI more adept at                │
│  understanding, interpreting, and generating human language.                                                    │
│  3. **Computer Vision**: The capacity of AI systems to recognize and understand visual information is           │
│  advancing rapidly, paving the way for applications like autonomous vehicles and facial recognition             │
│  technology.                                                                                                    │
│  4. **Emerging trends and applications in various industries**: From healthcare to finance, transportation,     │
│  retail, and beyond, AI is making a profound impact on 



Tasks are finished in 32.54870781000136 seconds.




╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 66b24579-6350-4d3a-85ee-b975ff08572e                                                                     │
│  Agent: Editor                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 1b4d43f8-5d88-4588-8961-646a06a4f7f0                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/1b4d43f8-5d88-458 │
│ 8-8961-646a06a4f7f0?access_code=TRACE-169993c99b                             │
│ 🔑 Access Code: TRACE-169993c99b                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d9a45f84-7a8a-4927-855e-7a8405b9d152                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Here's an edited version of the blog post that aligns with the brand's voice and follows         │
│  journalistic best practices:                                                                                   │
│                                                                                                                 │
│  ```markdown                                                                                                    │
│  # Unveiling the Latest in Artificial Intelligence: Trends, Pioneers, and News                                  │
│                                                                                                                 │
│  Welcome to our latest exploration into the dynamic world of Artificial Intelligence (AI)! As AI continues to   │
│  transform industries and reshape our daily lives, we delve deeper into the exciting advancements, key          │
│  players, and noteworthy news shaping this rapidly evolving field.                                              │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  Artificial Intelligence, a marvel of human innovation, is becoming increasingly integral to our society. With  │
│  breakthroughs in quantum computing, AI's potential has never been greater. Join us as we unravel the latest    │
│  trends, pioneers, and news in the realm of Artificial Intelligence.                                            │
│                                                                                                                 │
│  ## Latest Trends in AI                                                                                         │
│                                                                                                                 │
│  The cutting-edge research in AI is nothing short of breathtaking! From Machine Learning (ML) to Natural        │
│  Language Processing (NLP), Computer Vision, and beyond, here's a glimpse:                                      │
│                                                                                                                 │
│  1. **Machine Learning**: The subfields of Deep learning and Reinforcement learning are revolutionizing the     │
│  way machines learn from data.                                                                                  │
│  2. **Natural Language Processing (NLP)**: Recent advancements in NLP have made AI more adept at                │
│  understanding, interpreting, and generating human language.                                                    │
│  3. **Computer Vision**: The capacity of AI systems to recognize and understand visual information is           │
│  advancing rapidly, paving the way for applications like autonomous vehicles and facial recognition             │
│  technology.                                                                                                    │
│  4. **Emerging trends and applications in various indu

- Display the results of your execution as markdown in the notebook.

In [18]:
from IPython.display import Markdown, display

In [19]:
markdown_text = result.raw.strip()
if markdown_text.startswith("```"):
    lines = markdown_text.splitlines()
    if len(lines) > 2 and lines[-1].strip() == "```":
        markdown_text = "\n".join(lines[1:-1])
display(Markdown(markdown_text))

Here's an edited version of the blog post that aligns with the brand's voice and follows journalistic best practices:

```markdown
# Unveiling the Latest in Artificial Intelligence: Trends, Pioneers, and News

Welcome to our latest exploration into the dynamic world of Artificial Intelligence (AI)! As AI continues to transform industries and reshape our daily lives, we delve deeper into the exciting advancements, key players, and noteworthy news shaping this rapidly evolving field.

## Introduction

Artificial Intelligence, a marvel of human innovation, is becoming increasingly integral to our society. With breakthroughs in quantum computing, AI's potential has never been greater. Join us as we unravel the latest trends, pioneers, and news in the realm of Artificial Intelligence.

## Latest Trends in AI

The cutting-edge research in AI is nothing short of breathtaking! From Machine Learning (ML) to Natural Language Processing (NLP), Computer Vision, and beyond, here's a glimpse:

1. **Machine Learning**: The subfields of Deep learning and Reinforcement learning are revolutionizing the way machines learn from data.
2. **Natural Language Processing (NLP)**: Recent advancements in NLP have made AI more adept at understanding, interpreting, and generating human language.
3. **Computer Vision**: The capacity of AI systems to recognize and understand visual information is advancing rapidly, paving the way for applications like autonomous vehicles and facial recognition technology.
4. **Emerging trends and applications in various industries**: From healthcare to finance, transportation, retail, and beyond, AI is making a profound impact on numerous sectors, changing the way we live and work.
5. **AI ethics and regulations**: As AI becomes more ubiquitous, ethical considerations and regulatory measures, such as data privacy and bias mitigation, are growing in importance.

## Key Players in the AI Industry

Leading companies developing AI technologies are pushing the boundaries of what's possible:

1. **Google DeepMind**: Known for its work in reinforcement learning, Google DeepMind has made significant strides in mastering complex games like Go and chess.
2. **IBM Watson**: IBM Watson's AI platform is applied in a wide array of applications, from healthcare diagnostics to customer service.
3. **Microsoft Azure**: Microsoft's cloud-based AI platform offers numerous services, including machine learning, speech recognition, and natural language processing.
4. **Amazon Web Services (AWS)**: AWS provides a comprehensive suite of AI services, powering everything from Amazon's recommendation engine to Alexa, its voice assistant.
5. **NVIDIA**: A trailblazer in GPU technology, NVIDIA plays a crucial role in powering the hardware that fuels much of today's AI research and development.

Notable startups and research institutions are also making significant contributions to the field, with collaborations, partnerships, and acquisitions among these players defining the future of AI.

## Noteworthy News in AI

From breakthroughs to controversies, here's a roundup of recent news stories related to AI:

1. **AI Advancements**: Google's DeepMind made headlines with its advancement in protein folding, a significant milestone toward solving intricate medical problems using AI.
2. **AI-related Events**: The NeurIPS conference and the AI for Good Global Summit are just two examples of events where AI experts gather to discuss the latest developments and challenges in the field.
3. **Impact on Industry and Society**: The use of AI in agriculture has been instrumental in addressing food security issues, while concerns about job displacement due to AI remain a topic of ongoing debate.

## Call to Action

Stay informed about the latest trends in Artificial Intelligence by following relevant news sources. Engage in discussions about AI topics on online communities or social media platforms. For further reading, we recommend books, articles, and podcasts related to Artificial Intelligence. Together, let's embark on this fascinating journey of discovery!
```

This blog post is now well-written, aligned with the brand's voice, and follows journalistic best practices. The content is engaging, informative, and provides a balanced viewpoint on the topic of Artificial Intelligence.